In [49]:
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from datetime import date, timedelta

In [39]:
load_dotenv(override=True)
client = OpenAI()
MODEL = "gpt-5.4-nano"

In [40]:
NOTES = [
    "Dentist appointment moved to 3 PM",
    "Project ArchOS deadline approaching, to be kicked off soon in Q1",
    "Gift ideas for Jamka: toys, clothes, books"
]

TODOS: list[dict] = []

FX_RATES = {"USD": 1.0, "EUR": 0.93, "GBP": 0.81, "JPY": 144.0, "KGS": 87.0}

In [41]:
def get_weather(city: str, unit: str = "C") -> dict:
    fake_weather = {"chicago": 4, "bishkek": 10, "tokyo": 15, "munchen": 8, "paris": 12}
    temp_c = fake_weather.get(city.lower(), 20)
    temp = temp_c if unit.upper() == "C" else temp_c * 9 / 5 + 32
    return {"city": city, "temperature": temp, "unit": unit}

def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    src, dst = from_currency.upper(), to_currency.upper()
    
    if src not in FX_RATES or dst not in FX_RATES:
        return {"error": "Unsupported currency. Supported currencies are: " + ", ".join(FX_RATES.keys())}
    converted_amount = amount / FX_RATES[src] * FX_RATES[dst]
    return {"amount": converted_amount, "from": src, "to": dst}

def search_notes(query: str) -> dict:
    hits = [n for n in NOTES if query.lower() in n.lower()]
    return {"query": query, "matches": hits}

def add_todo(task: str, due_date: str | None = None) -> dict:
    item = {"id": len(TODOS) + 1, "task": task, "due_date": due_date}
    TODOS.append(item)
    return item

# Name: function lookup so executing a call is one line.
DISPATCH = {
    "get_weather": get_weather,
    "convert_currency": convert_currency,
    "search_notes": search_notes,
    "add_todo": add_todo
}
    

In [42]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city", 
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Name of the city, e.g., 'Chicago'"},
                "unit": {"type": "string", "enum": ["celcius", "fahrenheit"]},
            },
            "required": ["city"]
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "search_notes",
        "description": "Search for notes containing a specific query",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query for the notes"}
            },
            "required": ["query"]
        },
    },    
},
{
    "type": "function",
    "function": {
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string", "description": "ISO code, e.g., 'USD'"},
                "to_currency": {"type": "string", "description": "ISO code, e.g., 'EUR'"}
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "add_todo",
        "description": "Add a new todo item",
        "parameters": {
            "type": "object",
            "properties": {
                "task": {"type": "string"},
                "due_date": {
                    "type": "string",
                    "description": "Due date for the task, e.g., 'YYYY-MM-DD'"
                },
            },
            "required": ["task"]
        },
    },
},
]

In [ ]:
# The loop: ask, run whatever tools the model chose, ask agin

def ask(question: str, max_rounds: int = 5) -> str:
    today = date.today()
    messages = [
        {
            "role": "system",
            "content": (f"""
            You are a helpful assistant. Today's date is {today.isoformat()}.
            Tomorrow is {(today + timedelta(days=1)).isoformat()}
            "Use tools when they help; answer directly when they don't
            """
            ),
        },
        {"role": "user", "content": question}
    ]

    for _ in range(max_rounds):
        response = client.chat.completions.create(
            model= MODEL_NAME,
            messages=messages,
            tools=tools,
            tool_choice="auto" # model decides: no tool, one tool, or multiple tools 
        )
        msg = response.choices[0].message
        messages.append(msg)
        
        for call in msg.tool_calls:
            fn_name = call.function.name
            fn_args = json.loads(call.function.arguments)
            print(f"Calling tool: {fn_name} with arguments: {fn_args}")
            
            try:
                result = DISPATCH[fn_name](**fn_args)
            except Exception as exc:
                print("error:", str(exc))
            
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })
            
    return "Stopped: too many tool rounds"
            
    

In [50]:
gr.ChatInterface(fn=ask).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/tkarmysh/tlnt-ai/talantasan-git/agents/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tkarmysh/tlnt-ai/talantasan-git/agents/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tkarmysh/tlnt-ai/talantasan-git/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tkarmysh/tlnt-ai/talantasan-git/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tkarmysh/tlnt-ai/talantasan-git/a